# **1. Migration to NEON**

In [ ]:
# ============================================================
# Migrate vaccination.db (SQLite) -> Neon PostgreSQL
# ============================================================

!pip install psycopg2-binary sqlalchemy --quiet

import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text
from google.colab import userdata

# ---- 1. CONFIG ----
SQLITE_PATH = "/content/vaccination.db"

# Setting up connection to NEON.tech
NEON_CONNECTION_STRING = userdata.get('NEON_CONNECTION_STRING')

# ---- 2. CONNECT TO BOTH DATABASES ----
sqlite_conn = sqlite3.connect(SQLITE_PATH)
pg_engine = create_engine(NEON_CONNECTION_STRING)

# ---- 3. GET EVERYTHING TO MIGRATE (tables AND views — views get materialized as
#         regular Postgres tables since a SQLite view's SQL body doesn't always
#         translate directly, and a flat table is just as queryable) ----
objects = pd.read_sql(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name;",
    sqlite_conn
)
names = objects["name"].tolist()
print(f"Found {len(names)} tables/views to migrate:", names)

# ---- 4. MIGRATE EACH ONE ----
row_counts = {}
for name in names:
    df = pd.read_sql(f'SELECT * FROM "{name}";', sqlite_conn)
    df.to_sql(name, pg_engine, if_exists="replace", index=False, method="multi", chunksize=5000)
    row_counts[name] = len(df)
    print(f"{name:35s} {len(df):>8,} rows migrated")

sqlite_conn.close()

# ---- 5. VERIFY ROW COUNTS MATCH ----
print("\nVerifying row counts in Neon Postgres:")
with pg_engine.connect() as conn:
    for name in names:
        pg_count = conn.execute(text(f'SELECT COUNT(*) FROM "{name}"')).scalar()
        status = "OK" if pg_count == row_counts[name] else "MISMATCH"
        print(f"{name:35s} sqlite={row_counts[name]:>8,}  postgres={pg_count:>8,}  [{status}]")

Found 34 tables/views to migrate: ['agg_country_year_summary', 'dim_antigen', 'dim_antigen_disease_map', 'dim_country', 'dim_disease', 'dim_intro_disease_map', 'dim_vaccine', 'dim_who_region', 'fact_cases', 'fact_cases_global', 'fact_cases_region', 'fact_coverage', 'fact_coverage_global', 'fact_coverage_region', 'fact_incidence', 'fact_incidence_global', 'fact_incidence_region', 'fact_vaccine_introduction', 'fact_vaccine_schedule', 'stg_cases', 'stg_country_year_summary', 'stg_coverage', 'stg_coverage_valid', 'stg_incidence', 'stg_intro', 'stg_schedule', 'pbi_cases', 'pbi_country_year_summary', 'pbi_coverage', 'pbi_coverage_vs_incidence', 'pbi_incidence', 'pbi_region_kpi', 'pbi_vaccine_introduction', 'pbi_vaccine_schedule']
agg_country_year_summary               9,024 rows migrated
dim_antigen                               69 rows migrated
dim_antigen_disease_map                   17 rows migrated
dim_country                              195 rows migrated
dim_disease                   

# **2. Adding Primary Keys**

In [ ]:
# ============================================================
# Recreate primary keys
# ============================================================

pk_definitions = {
    "dim_country": ["country_code"],
    "dim_who_region": ["who_region_code"],
    "dim_antigen": ["antigen_code"],
    "dim_disease": ["disease_code"],
    "dim_vaccine": ["vaccine_code"],
    "dim_antigen_disease_map": ["antigen_code", "disease_code"],
    "dim_intro_disease_map": ["vaccine_name", "disease_code"],
    "fact_coverage": ["coverage_id"],
    "fact_incidence": ["incidence_id"],
    "fact_cases": ["case_id"],
    "fact_vaccine_introduction": ["intro_id"],
    "fact_vaccine_schedule": ["schedule_id"],
    "agg_country_year_summary": ["country_code", "year"],
}

with pg_engine.begin() as conn:
    for table, pk_cols in pk_definitions.items():
        if table not in names:
            continue
        cols_sql = ", ".join(f'"{c}"' for c in pk_cols)
        try:
            conn.execute(text(f'ALTER TABLE "{table}" ADD PRIMARY KEY ({cols_sql});'))
            print(f"PK added: {table} ({cols_sql})")
        except Exception as e:
            print(f"Skipped PK for {table}: {e}")

PK added: dim_country ("country_code")
PK added: dim_who_region ("who_region_code")
PK added: dim_antigen ("antigen_code")
PK added: dim_disease ("disease_code")
PK added: dim_vaccine ("vaccine_code")
PK added: dim_antigen_disease_map ("antigen_code", "disease_code")
PK added: dim_intro_disease_map ("vaccine_name", "disease_code")
PK added: fact_coverage ("coverage_id")
PK added: fact_incidence ("incidence_id")
PK added: fact_cases ("case_id")
PK added: fact_vaccine_introduction ("intro_id")
PK added: fact_vaccine_schedule ("schedule_id")
PK added: agg_country_year_summary ("country_code", "year")


# **3. Adding Foreign Keys**

In [ ]:
# ============================================================
# Recreate foreign keys (referential integrity between dims and facts)
# ============================================================

fk_definitions = [
    ("dim_country", "who_region_code", "dim_who_region", "who_region_code"),
    ("fact_coverage", "country_code", "dim_country", "country_code"),
    ("fact_coverage", "antigen_code", "dim_antigen", "antigen_code"),
    ("fact_incidence", "country_code", "dim_country", "country_code"),
    ("fact_incidence", "disease_code", "dim_disease", "disease_code"),
    ("fact_cases", "country_code", "dim_country", "country_code"),
    ("fact_cases", "disease_code", "dim_disease", "disease_code"),
    ("fact_vaccine_introduction", "country_code", "dim_country", "country_code"),
    ("fact_vaccine_introduction", "who_region_code", "dim_who_region", "who_region_code"),
    ("fact_vaccine_schedule", "country_code", "dim_country", "country_code"),
    ("fact_vaccine_schedule", "vaccine_code", "dim_vaccine", "vaccine_code"),
    ("fact_vaccine_schedule", "who_region_code", "dim_who_region", "who_region_code"),
    ("dim_antigen_disease_map", "antigen_code", "dim_antigen", "antigen_code"),
    ("dim_antigen_disease_map", "disease_code", "dim_disease", "disease_code"),
    ("dim_intro_disease_map", "disease_code", "dim_disease", "disease_code"),
    ("agg_country_year_summary", "country_code", "dim_country", "country_code"),
]

with pg_engine.begin() as conn:
    for child_table, child_col, parent_table, parent_col in fk_definitions:
        if child_table not in names or parent_table not in names:
            continue
        fk_name = f"fk_{child_table}_{child_col}"
        try:
            conn.execute(text(f'''
                ALTER TABLE "{child_table}"
                ADD CONSTRAINT {fk_name}
                FOREIGN KEY ("{child_col}") REFERENCES "{parent_table}" ("{parent_col}");
            '''))
            print(f"FK added: {child_table}.{child_col} -> {parent_table}.{parent_col}")
        except Exception as e:
            print(f"Skipped FK {child_table}.{child_col}: {e}")

FK added: dim_country.who_region_code -> dim_who_region.who_region_code
Skipped FK fact_coverage.country_code: (psycopg2.errors.ForeignKeyViolation) insert or update on table "fact_coverage" violates foreign key constraint "fk_fact_coverage_country_code"
DETAIL:  Key (country_code)=(ABW) is not present in table "dim_country".

[SQL: 
                ALTER TABLE "fact_coverage"
                ADD CONSTRAINT fk_fact_coverage_country_code
                FOREIGN KEY ("country_code") REFERENCES "dim_country" ("country_code");
            ]
(Background on this error at: https://sqlalche.me/e/20/gkpj)
Skipped FK fact_coverage.antigen_code: (psycopg2.errors.InFailedSqlTransaction) current transaction is aborted, commands ignored until end of transaction block

[SQL: 
                ALTER TABLE "fact_coverage"
                ADD CONSTRAINT fk_fact_coverage_antigen_code
                FOREIGN KEY ("antigen_code") REFERENCES "dim_antigen" ("antigen_code");
            ]
(Background on this e

# **4. Adding Indexes**

In [ ]:
# ============================================================
# Useful indexes for query performance
# ============================================================

index_statements = [
    'CREATE INDEX IF NOT EXISTS idx_coverage_country_year ON fact_coverage(country_code, year);',
    'CREATE INDEX IF NOT EXISTS idx_coverage_antigen ON fact_coverage(antigen_code);',
    'CREATE INDEX IF NOT EXISTS idx_incidence_country_year ON fact_incidence(country_code, year);',
    'CREATE INDEX IF NOT EXISTS idx_incidence_disease ON fact_incidence(disease_code);',
    'CREATE INDEX IF NOT EXISTS idx_cases_country_year ON fact_cases(country_code, year);',
    'CREATE INDEX IF NOT EXISTS idx_cases_disease ON fact_cases(disease_code);',
    'CREATE INDEX IF NOT EXISTS idx_intro_country_year ON fact_vaccine_introduction(country_code, year);',
    'CREATE INDEX IF NOT EXISTS idx_schedule_country_year ON fact_vaccine_schedule(country_code, year);',
]

with pg_engine.begin() as conn:
    for stmt in index_statements:
        conn.execute(text(stmt))
print("Indexes created.")

Indexes created.


In [ ]:
# ============================================================
# Quick sanity-check query against the live Neon database
# ============================================================
test_df = pd.read_sql('SELECT * FROM "pbi_region_kpi" LIMIT 5;', pg_engine)
test_df

,who_region_name,who_region_code,year,avg_coverage,total_cases,avg_incidence
0,African Region,AFRO,1980,5.7,1642381.0,774.28
1,African Region,AFRO,1981,13.2,1791159.0,793.15
2,African Region,AFRO,1982,14.3,1775986.0,763.58
3,African Region,AFRO,1983,20.2,1697304.0,701.22
4,African Region,AFRO,1984,25.3,1344417.0,552.18


# **5. Connecting to live vaccinedb on NEON.tech**

In [ ]:
# ============================================================
# Live connection to Neon PostgreSQL
# ============================================================

!pip install psycopg2-binary sqlalchemy --quiet

import pandas as pd
from sqlalchemy import create_engine, text
from getpass import getpass

# Setting up connection to NEON.tech
NEON_CONNECTION_STRING = userdata.get('NEON_CONNECTION_STRING')

engine = create_engine(NEON_CONNECTION_STRING, pool_pre_ping=True)

# Quick connectivity test
with engine.connect() as conn:
    version = conn.execute(text("SELECT version();")).scalar()
print("Connected to Neon Postgres:", version)

# Helper for ad-hoc querying
def run_query(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params or {})

# Example: list every table now living on Neon
run_query("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

Connected to Neon Postgres: PostgreSQL 18.6 (2078fcb) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


,table_name
0,agg_country_year_summary
1,dim_antigen
2,dim_antigen_disease_map
3,dim_country
4,dim_disease
5,dim_intro_disease_map
6,dim_vaccine
7,dim_who_region
8,fact_cases
9,fact_cases_global


# **6. Querying or Manipulating the Database for LIVE update on PowerBI Dashboard or on streamlit app**

In [ ]:
# ============================================================
# Recursive interactive query runner with password gate for non-SELECT statements
# ============================================================

from google.colab import userdata
from sqlalchemy import text
import pandas as pd
from getpass import getpass
import re

STOP_PATTERN = re.compile(r"^\s*stop\s*$", re.IGNORECASE)

def classify_query(sql: str) -> str:
    """Returns 'DQL' for plain reads (SELECT/WITH/EXPLAIN/SHOW), 'OTHER' for anything else
    (INSERT, UPDATE, DELETE, CREATE, DROP, ALTER, TRUNCATE, GRANT, etc.)."""
    stripped = sql.strip().lstrip("(").strip()
    first_word = stripped.split(None, 1)[0].upper() if stripped else ""
    return "DQL" if first_word in ("SELECT", "WITH", "EXPLAIN", "SHOW") else "OTHER"


def run_user_query(engine):
    sql = input("Enter your SQL query (PostgreSQL syntax), or type 'stop' to end:\n")

    if STOP_PATTERN.match(sql):
        print("Stopped. No more queries will be run.")
        return

    query_type = classify_query(sql)

    if query_type == "OTHER":
        entered_password = getpass(
            "⚠️  This looks like a DDL/DML statement (not a plain SELECT/WITH). "
            "Enter the confirmation password to proceed: "
        )
        correct_password = userdata.get("QUERY_CONFIRM_PASSWORD")  # set in Colab Secrets
        if entered_password != correct_password:
            print("❌ Incorrect password — query NOT executed.\n")
            return run_user_query(engine)
        print("✅ Password verified — executing.")

    try:
        with engine.begin() as conn:  # commits on success, rolls back automatically on error
            if query_type == "DQL":
                result_df = pd.read_sql(text(sql), conn)
                display(result_df)
            else:
                result = conn.execute(text(sql))
                print(f"Statement executed successfully. Rows affected: {result.rowcount}")
    except Exception as e:
        print("Error running query:", e)

    print()  # blank line for readability between prompts
    return run_user_query(engine)


# Usage — run this cell once; it keeps prompting until you type 'stop'
run_user_query(engine)

Enter your SQL query (PostgreSQL syntax), or type 'stop' to end:
stop                      
Stopped. No more queries will be run.
